<a href="https://colab.research.google.com/github/rustam4ike/rag_assistant/blob/main/practice_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install llama-index llama-index-readers-file llama-index-embeddings-huggingface llama-index-llms-huggingface transformers torch accelerate bitsandbytes peft docx2txt docling

In [ ]:
!pip install llama-index-postprocessor-colbert-rerank

In [ ]:
from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
    PromptTemplate,
    Settings,
)
from docling.document_converter import DocumentConverter
from llama_index.core.graph_stores import SimpleGraphStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.postprocessor import LLMRerank, SentenceTransformerRerank
from llama_index.postprocessor.colbert_rerank import ColbertRerank
from peft import PeftModel, PeftConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, GenerationConfig
import torch
import re
import glob

In [ ]:
# ---ЗАГРУЗКА МОДЕЛИ---
# Путь на HuggingFace
MODEL_NAME = "IlyaGusev/saiga_mistral_7b"

# Определяем параметры квантования
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,                      # 4-битное квантование
    bnb_4bit_quant_type="nf4",              # Используем формат NF4
    bnb_4bit_use_double_quant=True,         # Применить повторное квантование
    bnb_4bit_compute_dtype=torch.float16    # Тип из которого преобразуем
)

# Создание конфига, соответствующего методу PEFT (в нашем случае LoRA)
config = PeftConfig.from_pretrained(MODEL_NAME)

# Загружаем базовую модель, ее имя берем из конфига для LoRA
model = AutoModelForCausalLM.from_pretrained(
    config.base_model_name_or_path,          # Идентификатор модели
    quantization_config=quantization_config, # Параметры квантования
    torch_dtype=torch.float16,               # Тип данных
    device_map="auto"                        # Автоматический выбор типа устройства
)

# Загружаем LoRA модель
model = PeftModel.from_pretrained(
    model,
    MODEL_NAME,
    torch_dtype=torch.float16
)

# Переводим модель в режим инференса
# Можно не переводить, но явное всегда лучше неявного
model.eval()

# Загружаем токенизатор
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

In [ ]:
# ---НАСТРОЙКА LlamaIndex---
# Параметры генерации
generation_config = GenerationConfig.from_pretrained(MODEL_NAME)    # Параметры по умолчанию
generation_config.max_new_tokens = 512                              # Вносим изменения
print(generation_config)

# Настройка языковой модели
Settings.llm = HuggingFaceLLM(
    model=model,                                                # Загруженная модель
    model_name=MODEL_NAME,                                      # Идентификатор модели
    tokenizer=tokenizer,                                        # Токенизатор
    max_new_tokens=generation_config.max_new_tokens,            # Макс.количество токенов
    generate_kwargs={"generation_config": generation_config}    # Параметры генерации
)

# Настройка эмбеддера
Settings.embed_model = HuggingFaceEmbedding(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"    # Модель для русского языка
)

# Настройка чанков
text_splitter = SentenceSplitter(
    chunk_size=512,     # Размер чанка в токенах
    chunk_overlap=50,   # Перекрытие между чанками
)

In [ ]:
# ---КОНВЕРТАЦИЯ ДАННЫХ---
converter = DocumentConverter()

# Находим все .docx файлы в текущей папке
docx_files = glob.glob("*.docx")

for docx_path in docx_files:
    # Конвертируем
    result = converter.convert(docx_path)
    markdown_text = result.document.export_to_markdown()

    # Сохраняем как .md
    md_path = docx_path.replace(".docx", ".md")
    with open(md_path, "w", encoding="utf-8") as f:
        f.write(markdown_text)

    print(f"✅ Сохранён: {md_path}")

✅ Сохранён: Анализ_АТТ.md
✅ Сохранён: Характеристика_АТТ.md


In [ ]:
# ---ЗАГРУЗКА ДАННЫХ---
md_path_text = "Характеристика_АТТ.md"
md_path_table = "Анализ_АТТ.md"

# Загружаем документы из MD
text_docs = SimpleDirectoryReader(input_files=[md_path_text]).load_data()
table_docs  = SimpleDirectoryReader(input_files=[md_path_table]).load_data()

print(f"📝 Пример текста из документов:")
print(text_docs[0].text[:500])
print(table_docs[0].text[:500])

📝 Пример текста из документов:
**2. АНАЛИЗ ПРОИЗВОДСТВЕННО-ФИНАНСОВОЙ ДЕЯТЕЛЬНОСТИ ПРЕДПРИЯТИЯ**

**2.1. Характеристика предприятия**

Общество с ограниченной ответственностью "Нижневартовское управление технологического транспорта № 1" (далее - Общество) создано в соответствии с Гражданским кодексом Российской Федерации и Федеральным законом Российской Федерации "Об обществах с ограниченной ответственностью".

Общество является юридическим лицом и осуществляет свою деятельность в соответствии с законодательством Российской Ф
**Подвижной состав**

| **Марка машины**   |   **Количество АТС** |
|--------------------|----------------------|
| 1. КамАЗ           |                   95 |
| 2. Урал            |                  159 |
| 3. КрАЗ            |                   15 |
| 4. Маз             |                   10 |
| 5. ЗиЛ             |                   16 |
| 6. Татра           |                   16 |
| 7. Газ             |                   15 |
| 8. Ваз             |          

In [ ]:
# ---НАСТРОЙКА ПРОМПТА ДЛЯ SAIGA---
text_prompt = PromptTemplate(
    """<s>system
    Ты специалист знающий всё о предприятии!
    Отвечай максимально точно по документу, не придумывай ничего от себя.
    Ответ должен быть четким, без лишней информации.
    </s>
    <s>user
    Источник: {context_str}
    Вопрос: {query_str}
    </s>
    <s>bot
    """
)

table_prompt = PromptTemplate(
    """<s>system
    Ты главный экономист предприятия!
    Финансовый директор может задать вопросы по анализу данных.
    Отвечай максимально точно по документу, не придумывай ничего от себя.
    Ответ должен быть четким и без лишней информации,
    не придумывай никаких цифр или названий от себя.
    Если данные не найдены в таблице, скажи: "Данные отсутствуют".
    Если нужно будет вычисление результатов, дай точное решение по ним.
    </s>
    <s>user
    Источник: {context_str}
    Вопрос: {query_str}
    </s>
    <s>bot
    """
)

In [ ]:
# ---НАСТРОЙКИ РАНЖИРОВАНИЯ---
# Метод ColBERT
colbert_reranker = ColbertRerank(
    top_n=2,                              # Количество лучших чанков
    model="colbert-ir/colbertv2.0",
    tokenizer="colbert-ir/colbertv2.0",
    keep_retrieval_score=True,
)

# Метод LLMRerank
llm_reranker = LLMRerank(
            llm=Settings.llm,
            choice_batch_size=5,
            top_n=1
)

In [ ]:
# ---СОЗДАНИЕ ИНДЕКСОВ---
text_index = VectorStoreIndex.from_documents(text_docs)
table_index = VectorStoreIndex.from_documents(table_docs)

In [ ]:
# ---ДВИЖКИ ЗАПРОСОВ---
# Для текстовых данных
text_engine = text_index.as_query_engine(
    include_text=True,
    text_qa_template=text_prompt,                           # Правильный способ задать промпт
    similarity_top_k=10,                                    # Ищем самые релевантные куски
    verbose=True
)

# Для табличных данных
table_engine = table_index.as_query_engine(
    include_text=True,
    text_qa_template=table_prompt,
    similarity_top_k=10,
    verbose=True,
    node_postprocessors=[colbert_reranker, llm_reranker]    # Указываем метод ранжирования
)


In [ ]:
# --- ПРОСТОЙ ФИЛЬТР---
class SimpleGuardrails:
    def __init__(self):
        self.blocked_words = [
            r"оружи[ею]", r"фаши[сз]", r"раси[сз]",
            r"алкогол[иь]", r"террор[и]"
        ]

    def check(self, query):
        query_lower = query.lower()
        for word in self.blocked_words:
            if re.search(word, query_lower):
                return False, f"Запрос содержит запрещенное слово: '{word}'"
        return True, query

In [ ]:
# ---ФУНКЦИЯ ТЕСТИРОВАНИЯ---
guardrails = SimpleGuardrails()

def test_query(query, guardrails, engine):
    print(f"\n{'='*50}")
    print(f"Вопрос: {query}")
    print(f"{'='*50}")

    # Проверка через фильтр
    is_allowed, result = guardrails.check(query)

    if not is_allowed:
        print(f"⛔ {result}")
        return result

    # Отправка в LlamaIndex
    response = engine.query(query)
    answer = str(response).strip()
    answer = answer.replace('<s>bot', '').replace('</s>', '').strip()
    print(f"Ответ: {answer}\n")
    return answer

In [ ]:
# ---ТЕСТЫ---
# Вопросы для текстовых данных
text_queries = [
    "Какая численность техники на предприятии",
    "Какой уставной капитал общества",
    "Кто главный в компании",
    "Сколько колонн на предприятии",
    "Где расположенно предприятие и напиши его сокращенное название"
]
# Вопросы для табличных данных
table_queries = [
    "Напиши прибыль от реализации план и факт, и рассчитай разницу",
    "Напиши расходы на ГСМ план и факт, и рассчитай разницу",
    "Выведи данные из таблицы подвижного состава"
]

In [ ]:
# Вывод ответов в зависимости от движка запросов
for query in text_queries:
    test_query(query, guardrails, text_engine)


Вопрос: Какая численность техники на предприятии
Ответ: 572 единицы техники.


Вопрос: Какой уставной капитал общества
Ответ: 10 000 (десять тысяч).


Вопрос: Кто главный в компании
Ответ: Главным в компании является Генеральный директор. Он отвечает за все вопросы, связанные с функционированием компании, включая принятие решений, планирование, контроль и управление ресурсами.


Вопрос: Сколько колонн на предприятии
Ответ: 10 колонн.


Вопрос: Где расположенно предприятие и напиши его сокращенное название
Ответ: 1. ООО «НУТТ-1» расположено в городе Нижневартовске, Тюменской области, Ханты-Мансийском автономном округе-Югра.

2. Сокращенное название предприятия - НУТТ-1.



In [ ]:
# Вывод ответов в зависимости от движка запросов
for query in table_queries:
    test_query(query, guardrails, table_engine)


Вопрос: Напиши прибыль от реализации план и факт, и рассчитай разницу


/usr/local/lib/python3.12/dist-packages/llama_index/postprocessor/colbert_rerank/base.py:120: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  node.score = float(score)


Ответ: 1. Прибыль от реализации по плану: 371 718 - 368 781 = 2 937
    2. Прибыль от реализации факт: 424 268 - 421 859 = 2 409
    3. Разница: 2 937 - 2 409 = 528

Вывод: прибыль от реализации по факту составляет 528 тысяч рублей больше, чем по плану.


Вопрос: Напиши расходы на ГСМ план и факт, и рассчитай разницу
Ответ: 1.1. ГСМ план: 45 549,0
    1.1. ГСМ факт: 61 579,0
    1.1. Разница: 16 030,0


Вопрос: Выведи данные из таблицы подвижного состава
Ответ: 1. КамАЗ - 95
    2. Урал - 159
    3. КрАЗ - 15
    4. Маз - 10
    5. ЗиЛ - 16
    6. Татра - 16
    7. Газ - 15
    8. Ваз - 14
    9. Остальные - 220

Всего - 572

1. КамАЗ - 13.4180790960
2. Урал - 27.7972027972
3. КрАЗ - 2.62237762237
4. Маз - 1.74825174825
5. ЗиЛ - 2.7972027972
6. Татра - 2.79720279720
7. Газ - 2.62237762238
8. Ваз - 2.44755244755
9. Остальные - 38.4615384615

Всего - 100%

**Производственная программа на 1 квартал 2009 года:**

| № пп     | Заказчик                 | 1 кв        | 2 кв        | 3 кв     

ПУНКТЫ ЗАДАНИЯ:
1. В качестве нейро-сотрудников выступают специалист и главный экономист предприятия, первый отвечает за основную информацию, а второй за технико-экономические показатели
2. В качестве базы знаний использована моя дипломная работа еще со студенческих годов, с настоящей информацией за 2008 и 2009 года
3. Использован фреймворк LlamaIndex
4. Задействована LLM saiga_mistral_7b_lora
5. Трассировка не используется, ввиду конфликта библиотек
6. Галлюинации обнаружены, и исправлены
7. Смотрите пункт шестой
8. Задействованы приемы каскадного ранжирования
9. Для безопасноти запросов создан класс с простой фильтрацией
________________________________________________________________________________

АЛГОРИТМ ВЫПОЛНЕНИЯ:

В данной практической работе использован стек технологий тот же что и в уроках 30 и 31, так как он позволяет грамотно использовать загрузку модели saiga_mistral_7b c рускоязычным адаптером LoRA. Задействован фреймворк Llamma с настройкой языковой модели, эмбеддеров и чанков. Проведена конвертация файлов docx в структурированый тип MD, с последующей их загрузкой. Применны настройки ранжирования ColBERT в сочетании с двухэтапным LLMRerank. Созданы два индекса, два движка запросов для текстовых и табличных данных. Применен простой фильтр с ключевыми словами. В конце задания создана функция тестирования, в которой имеется выбор движка запроса в зависимости от цели вопроса.
________________________________________________________________________________

РАСШИФРОВКА ПУНКТОВ:

Пунты с 1 по 3 не требуют объяснения, так как информация в них уже изложена.

Пункт 4. Помимо модели "saiga_mistral_7b_lora", рассматривал вариант уже с квантованной версией "ruGPT-3.5-13B-4bit-bnb". Обе модели достаточно хороши в использовании, но в случае потребности использования графа знаний saiga в скорости намного превосходит ruGPT, поэтому выбор очевиден.

Пункт 5. Трассировку с помощью феникс запустить не удалось, ввиду конфликта библиотек и зависимостей.

Пункт 6-8. Изначально я использовал один общий файл где была отражена характеристика предприятия и анализ его показателей. Первый опыт показал то если текстовые данные и табличные использовать как одно целое, это неправильно и модель будет точно галлюционировать. После разбивки на два файла, ситуация улучшилась, но не сильно. Из всех 12 болевых точек RAG, мне самым интерсным показалось метод ранжирования. Проведя несколько эксперементов с тремя вариантами, самым лучшим оказалось использование каскадного метода, а именно сначала запущен ColBERT,а потом уже LLMRerank, который выбрал лучшие чанки от предыдущего, так как сам ColBERT не идеально подбирал лучший вариант.

Пункт 9 так же не требует более подробного объяснения.